# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [9]:
# Import necessary libraries
import sys
!{sys.executable} -m pip install scikit-learn
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 42.1 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


## Model Choice

Baseline Model: Naive Linear Regression (OLS)
For the baseline, I will use an Ordinary Least Squares (OLS) linear regression. In causal inference, it is standard practice to first run a simple regression of the target variable (re78) on the treatment indicator (treat) and the confounding covariates. This provides a baseline Average Treatment Effect (ATE) estimate. However, because OLS struggles with severe non-linear confounding and lack of overlap (as discovered in our EDA), this baseline could be inaccurate.I will compare this baseline against more robust causal models later.


## Feature Selection

For the baseline model, I will be utilizing all available features in the dataset.

Selected Features:

* Treatment Indicator: treat

* Confounding Covariates: age, educ, married, nodegree, re74, re75, race_hispan, and race_white (derived from one-hot encoding).

Justification:
In standard predictive machine learning, feature selection is often used to remove noisy or redundant variables to prevent overfitting. However, my primary goal here is causal inference. To accurately estimate the treatment effect, we must satisfy the conditional independence assumption (unconfoundedness). This means we must control for all observable variables that could influence both a person's likelihood of receiving the training and their future earnings.
As discovered during the Exploratory Data Analysis (EDA), features like historical earnings (re74, re75), education (educ, nodegree), and demographics (married, race) show severe imbalances between the treatment and control groups. Dropping any of these covariates would result in omitted variable bias, making the baseline treatment effect estimate completely invalid. Since the feature space is relatively small (9 features for 614 observations), keeping all features will not cause the "curse of dimensionality" and is necessary for robust causal modeling.


In [10]:
# 1. Load the dataset directly from the web
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/MatchIt/lalonde.csv"
df = pd.read_csv(url)

# Drop the extra index column from R
if 'rownames' in df.columns:
    df = df.drop(columns=['rownames'])

# 2. Feature Engineering: One-Hot Encoding the 'race' column
# Machine learning models require numbers, not text like 'black' or 'white'
df = pd.get_dummies(df, columns=['race'], drop_first=True, dtype=int)

# 3. Feature and target variable selection
# We want all columns EXCEPT the target variable to be our features (X)
X = df.drop(columns=['re78'])
y = df['re78']

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (491, 9)
Testing data shape: (123, 9)


## Implementation
The OLS model is implemented without hyperparameter tuning as it is not necessary.

1. The Baseline Effect
The model estimates that the training program boosted 1978 earnings by $1,378.80. However, the p-value is 0.080. Since this is higher than the standard 0.05 cutoff, the treatment effect is technically not statistically significant in this simple model.

2. Model Fit
The R-squared is 0.174, meaning this model only explains about 17.4% of the variation in 1978 earnings. Human income is complicated, and a simple linear equation just doesn't capture the whole picture!

3. The Biggest Predictors
Just like I saw in my EDA, past earnings matter the most. 1974 earnings are highly significant (p < 0.001). Additionally, the race_white demographic feature showed a statistically significant positive impact on 1978 earnings in this specific model setup.

4. The Red Flags
The output threw a warning about strong multicollinearity (which makes sense, since variables like educ and nodegree are basically measuring the same thing). Because OLS struggles with these overlapping features and the heavy selection bias I found earlier, I know I can't fully trust this $1,378.80 estimate.



In [11]:
# Implement the Baseline Model using statsmodels for statistical summary
# We rejoin X_train and y_train temporarily to use the formula API
train_data = pd.concat([X_train, y_train], axis=1)

# Define the regression formula: re78 ~ treat + age + educ + ... (all other features)
features_str = " + ".join(X_train.columns)
formula = f"re78 ~ {features_str}"

# Fit the OLS Baseline Model
baseline_model = smf.ols(formula, data=train_data).fit()

# Print the statistical summary
print(baseline_model.summary())

# The coefficient for 'treat' is our Baseline Average Treatment Effect (ATE)
print("\n--- Baseline Treatment Effect ---")
print(f"Estimated effect of training on 1978 earnings: ${baseline_model.params['treat']:.2f}")

                            OLS Regression Results                            
Dep. Variable:                   re78   R-squared:                       0.174
Model:                            OLS   Adj. R-squared:                  0.158
Method:                 Least Squares   F-statistic:                     11.22
Date:                Wed, 29 Jul 2026   Prob (F-statistic):           5.59e-16
Time:                        15:17:53   Log-Likelihood:                -4989.2
No. Observations:                 491   AIC:                             9998.
Df Residuals:                     481   BIC:                         1.004e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept     -36.8622   2501.500     -0.015      

## Evaluation

Evaluation MetricsTo evaluate my baseline model and set a solid benchmark for the advanced causal models I will build later, I am using a combination of traditional machine learning metrics and causal inference statistics:
* Average Treatment Effect (ATE) & p-value: Since my main goal is causal inference, I am heavily focused on the coefficient of the treat variable. This tells me the estimated dollar impact of the training program, while the p-value tells me if that finding is statistically significant.
* Root Mean Squared Error (RMSE): I am using RMSE because it heavily penalizes large prediction errors. Since my dataset has some massive high-earning outliers, I need to know when my model gets those drastically wrong.
* Mean Absolute Error (MAE): I chose MAE because it gives me a very straightforward, easy-to-understand metric: the average dollar amount my predictions are off by.
* R-squared ($R^2$): I am using this to measure how much of the variance in 1978 earnings is actually explained by my features, helping me understand if I am capturing the true complexity of the data.



In [12]:
# Predict the 1978 earnings on our unseen test set
y_pred = baseline_model.predict(X_test)

# Calculate the regression metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # RMSE is just the square root of MSE
mae = mean_absolute_error(y_test, y_pred)

# Print the results nicely formatted
print("--- Baseline Model Evaluation ---")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"Mean Absolute Error (MAE):      ${mae:,.2f}")
print(f"Mean Squared Error (MSE):       {mse:,.2f}")

--- Baseline Model Evaluation ---
Root Mean Squared Error (RMSE): $9,051.88
Mean Absolute Error (MAE):      $6,300.31
Mean Squared Error (MSE):       81,936,618.64
